In [7]:
!pip install torch_geometric transformers

In [8]:
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import SAGEConv, global_mean_pool
from transformers import AutoModel, AutoTokenizer

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

class GraphSAGEClassifier(nn.Module):
    def __init__(self, in_dim=32, hidden_dim=64, num_classes=8, num_layers=3):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.convs = nn.ModuleList([SAGEConv(in_dim, hidden_dim)])
        for _ in range(num_layers - 1):
            self.convs.append(SAGEConv(hidden_dim, hidden_dim))
        self.classifier = nn.Linear(hidden_dim, num_classes)

    def forward(self, x, edge_index, batch, return_embedding=False):
        for conv in self.convs:
            x = F.relu(conv(x, edge_index))
        g = global_mean_pool(x, batch)
        logits = self.classifier(g)
        if return_embedding:
            return logits, g
        return logits

class BertTextEncoder_T3(nn.Module):
    def __init__(self):
        super().__init__()
        self.bert = AutoModel.from_pretrained("distilbert-base-uncased")
        self.proj = nn.Linear(self.bert.config.hidden_size, 128)

    def forward(self, texts, tokenizer):
        enc = tokenizer(texts, truncation=True, padding=True, max_length=32, return_tensors="pt")
        enc = {k: v.to(device) for k, v in enc.items()}
        out = self.bert(**enc)
        return out.last_hidden_state, self.proj(out.last_hidden_state[:, 0, :]), enc["attention_mask"]

class CrossAttentionFusion_T3(nn.Module):
    def __init__(self):
        super().__init__()
        self.graph_proj = nn.Linear(64, 128)
        self.q_proj = nn.Linear(128, 128)
        self.k_proj = nn.Linear(768, 128)
        self.v_proj = nn.Linear(768, 128)
        self.classifier = nn.Linear(256, 8)
        self.scale = 128 ** 0.5

    def forward(self, g_raw, token_embeddings, attention_mask):
        g = self.graph_proj(g_raw)
        Q = self.q_proj(g).unsqueeze(1)
        K = self.k_proj(token_embeddings)
        V = self.v_proj(token_embeddings)
        scores = (Q @ K.transpose(-2, -1)) / self.scale
        mask = attention_mask.unsqueeze(1).bool()
        scores = scores.masked_fill(~mask, float("-inf"))
        attn = torch.softmax(scores, dim=-1)
        attended_text = (attn @ V).squeeze(1)
        z = torch.cat([g, attended_text], dim=-1)
        return self.classifier(z)

print("Initializing Encoders and Fusion Head...")
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

gnn = GraphSAGEClassifier().to(device)
bert = BertTextEncoder_T3().to(device)
fusion = CrossAttentionFusion_T3().to(device)

print("Loading Trained Weights...")
gnn.load_state_dict(torch.load("task2_gnn_graphsage.pt", map_location=device))
t3_checkpoint = torch.load("task3_fusion_model.pt", map_location=device)
fusion.load_state_dict(t3_checkpoint["fusion_model"], strict=False)
bert.load_state_dict(t3_checkpoint["bert_encoder"], strict=False)

gnn.eval()
bert.eval()
fusion.eval()

genre_list = ['Electronic', 'Experimental', 'Folk', 'Hip-Hop', 'Instrumental', 'International', 'Pop', 'Rock']
text_prompt = "A heavy, distorted electric guitar riff with fast acoustic drums."

print(f"\n[Input Text Context]: '{text_prompt}'")
print("[Input Audio Graph]: Loading sample graph structure...")

try:
    sample_graph = torch.load("graph_0.pt", map_location=device, weights_only=False)
    batch = torch.zeros(sample_graph.x.shape[0], dtype=torch.long).to(device)
    with torch.no_grad():
        _, g_emb = gnn(sample_graph.x.to(device), sample_graph.edge_index.to(device), batch, return_embedding=True)
    print("-> Audio graph processed successfully via GraphSAGE.")
except FileNotFoundError:
    print("-> graph_0.pt not found! Using simulated graph embedding fallback.")
    g_emb = torch.randn(1, 64).to(device)

with torch.no_grad():
    token_emb, cls_proj, mask = bert([text_prompt], tokenizer)
    logits = fusion(g_emb, token_emb, mask)
    prediction = genre_list[logits.argmax(dim=1).item()]

print("\n========================================")
print("END-TO-END INFERENCE RESULT")
print("========================================")
print(f"Predicted Musical Context: {prediction}")

Initializing Encoders and Fusion Head...


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_transform.bias    | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading Trained Weights...

[Input Text Context]: 'A heavy, distorted electric guitar riff with fast acoustic drums.'
[Input Audio Graph]: Loading sample graph structure...
-> graph_0.pt not found! Using simulated graph embedding fallback.

END-TO-END INFERENCE RESULT
Predicted Musical Context: Instrumental
